# Exemplo de pipeline ETL

Carrega dados não estruturados de um site web para uma tabela SQL em postgreSQL

Ela foi implementada usando tecnicas de Web Crawling com Selenium, aplicando um mecanismo de RPA (robot process automation) para baixar anuncios selecionados do OLX que estão paginados e espalhados no site.

## requerimentos:
### instalar geckodriver para o navegador que se quer usar, p.e.:
sudo apt install firefox-geckodriver

### instalar o selenium:
pip install selenium

### instalar conector do postgreSQL para python:
pip install psycopg2-binary



## migration - tabela necessária para conter os anúncios em formato raw

    CREATE DATABASE RPA_leo;

    CREATE ROLE rpa NOSUPERUSER NOCREATEDB NOCREATEROLE NOINHERIT LOGIN NOREPLICATION NOBYPASSRLS PASSWORD 'rpa';

    GRANT ALL ON TABLE public.anuncios_veiculos TO rpa;

    CREATE TABLE IF NOT EXISTS  anuncios_veiculos (
        id_anuncio serial primary key,
        cod_anuncio varchar(50),
        descricao varchar(255),
        valor_txt varchar(50),
        ano_txt varchar(20),
        km_txt varchar(20),
        link_txt varchar(256)
    );


### Import de bibliotecas

In [18]:
from selenium import webdriver

import psycopg2
import time
import sys

### Definição de parâmetros globais

In [19]:
%run globais.ipynb

# Funções

In [20]:
# Le a pagina atual, carrega os elementos desejados, verifica se tem mais páginas e avança antes de retornar
# Se não tiver mais página, retorna 0, senao retorna numero proxima pagina

def LePagina(driver, url_pagina_inicial, page):

    lstRetorno = []


    driver.get(url_pagina_inicial+'&o='+str(page))


    lstCards = driver.find_elements("xpath","//section[@class='olx-adcard  olx-adcard__vertical undefined']")

    if len(lstCards)>0:
       bAchou = True
       print("Achei ",len(lstCards),"anúncios na página")
    else:
       print("Não Achei anuncios")
       bAchou = False

    for card in lstCards:

        #pega o preco do anuncio
        try:


            preco_div = card.find_element("xpath","./div/div[@class='olx-adcard__mediumbody']")
            #preco_elem = card.find_element("xpath","./div/div/h3[@class='typo-body-large olx-adcard__price font-semibold']")
            if preco_div:
                preco = preco_div.text
        except:
            preco = "0"


        #Pega km e ano do anuncio
        try:
            km = ""
            ano = ""

            #testa primeiro se tem esses elementos
            det_elem = card.find_element("xpath","./div/div/div/div/div[@class='olx-adcard__detail']")
            #se nao der erro, continua mas pega a lista com todos
            if det_elem:
                lstDetails = card.find_elements("xpath","./div/div/div/div/div[@class='olx-adcard__detail']")
                for d in lstDetails:
                    if "km" in d.text:
                        km = d.text
                    elif "Ano" in d.get_attribute("aria-label"):
                        ano = d.get_attribute("aria-label")
        except:
            km = ""
            ano = ""


        #print(f"km={km} e ano={ano}")

        #pega o link do anuncio
        try:
            link_elem = card.find_element("xpath","./div/div/a[@class='olx-adcard__link']")
            if link_elem:
                href = link_elem.get_attribute("href")
                cod_anuncio = href.split("-")[-1]
                titulo = link_elem.get_attribute("title")
        except:
            cod_anuncio = "0"
            href = ""
            titulo = ""

        lstRetorno.append([cod_anuncio, titulo[0:255], preco ,ano,km,href])

        print("anuncio:",titulo, ano, km, preco)

    #verifica se tem outra página
    try:
        paginacao = driver.find_element("id","listing-pagination")

        prox = paginacao.find_element("xpath","./div/a[text()='Próxima página']")

        #verifica se está habilitado
        class_name = prox.get_attribute("class")
        if "olx-core-button--disabled" not in class_name:
            temProxima = True
        else:
            temProxima = False
    except:
        temProxima = False

    if (temProxima):
        print("Tem mais página")
        page += 1
        return page,lstRetorno
    else:
        return 0,lstRetorno


In [21]:
def CarregaDadosOLX( maxPages=1):
    driver = webdriver.Firefox()
    driver.implicitly_wait(30) # seconds
    print("Abrindo pagina inicial")
    lstAnuncios = []

    try:

        page = 1
        pagina_inicial = "https://www.olx.com.br/autos-e-pecas/motos/estado-sc?ic=branding&local=banner-descubra&campaign=motoshmautos&message=home"
        driver.get(pagina_inicial)

        print("\n verificar se já conseguiu abrir a página:")

        #x = input("Terminou de logar (S/N):")
    except:
        #print("*****  ERRO  execução captura web com RPA Selenium ****")
        print("*****  ERRO  execução da primeira página com Selenium ****")
        x = input("Continuar e fechar a página após erro?")


    try:

        div_total = driver.find_element("id","total-of-ads")

        cookie = driver.find_element("id","adopt-accept-all-button")

        if (cookie):

            cookie.click()    # aceita os cookies da página


        #Estabelecida a sessão corretamente

        #Lê cada pagina agora

        print ("inicio da execução")

        page,lstRetorno = LePagina(driver, pagina_inicial, 1)
        lstAnuncios.extend(lstRetorno)

        while (page>0):

            #so para controle inicial
            if(page > maxPages): break

            print("Pagina:",page)


            page,lstRetorno = LePagina(driver, pagina_inicial, page)
            lstAnuncios.extend(lstRetorno)
            #print(lstRetorno)

            #repete o ciclo

        print("Fim da captura")
    except:
        #print("*****  ERRO  execução captura web com RPA Selenium ****")
        print("*****  ERRO  execução captura web com RPA Selenium ****")
        x = input("Continuar e fechar a página após erro?")

    finally:
        driver.quit()



    return lstAnuncios




In [22]:
# Conexao com o banco de dados

def ConectaBD_JeoLab():

    print('Conectando com BD do RPA_leo para salvar no banco de dados do ADW...')

    ##### PostGres de LAB
    host = DB_HOST   #usa port-forward do kubectl para acessar ambiente azure
    port = 5432          #usa port-forward do kubectl para acessar ambiente azure
    user = DB_USER
    dbname = DB_DBNAME
    password= DB_PASSWORD


    conn_string = "host='"+host+"' dbname='"+dbname+"' user='"+user+"' password='"+password+"'"

    # print the connection string we will use to connect
    print ("Connecting to database <"+host+":"+str(port)+"> ...")

    #get a connection, if a connect cannot be made an exception will be raised here
    conn = psycopg2.connect(dbname=dbname, host=host, user=user, password=password, port=port)

    print ("Connected!\n")

    return conn

# Passo 1: Extratação de dados do site, na categoria escolhida

In [23]:
print("Baixando anuncios motos do OLX  - Santa Catarina")

lstAnuncios = []

MAX_PAGES = 2

Baixando anuncios motos do OLX  - Santa Catarina


In [26]:
lstAnuncios = CarregaDadosOLX(MAX_PAGES)


Abrindo pagina inicial

 verificar se já conseguiu abrir a página:
*****  ERRO  execução captura web com RPA Selenium ****


# Passo 2: Carga dos dados extraídos no BD estruturado com cada aúncio em registros RAW

In [28]:
if lstAnuncios and len(lstAnuncios) > 0:
    conn = ConectaBD_JeoLab()
    cursor = conn.cursor()

    print("BD conexão ok")

    #insere cada registro encontrado no anuncio no BD
    print("Inserindo registros lidos no BD JeoLAB - tabela OLX_motos")

    print("Ainda sem um mecanismo de update por diferença, deletando registros e reinserindo...")
    sql = 'DELETE FROM anuncios_veiculos where true'
    cursor.execute(sql)

    for a in lstAnuncios:
        sql = 'INSERT INTO rpa_leo.public.anuncios_veiculos (cod_anuncio, descricao, valor_txt, ano_txt, km_txt, link_txt) VALUES ('

        for c in a[:-1]:
            sql += "'"+c+"',"
        sql += "'"+a[-1]+"');"

        print(sql)
        # execute our Query
        cursor.execute(sql)

    conn.commit()
    conn.close()

# Próximos passos do projeto

## Passo 3: Limpeza e Transformação de dados raw em dados úteis
### usando mecanismos de processamento de textos ou mesmo NLP

## Passo 4: Carga nas tabelas estruturadas finais para uso em modelos de IA ou plataforma de dados agregados